In [118]:
from utils import get_dataset_lines

# Profile HMM Problem

Construct a profile HMM from a multiple alignment.

**Input**: A multiple alignment $Alignment$ and a threshold $\theta$.

**Output**: $HMM(Alignment, \theta)$, in the form of transition and emission matrices.

**Code Challenge**: Solve the Profile HMM Problem.

**Input**: A threshold $\theta$, followed by an alphabet $\Sigma$, followed by a multiple alignment $Alignment$ whose strings are formed from $\Sigma$.

**Output**: The transition matrix followed by the emission matrix of $HMM(Alignment, \theta)$.

**Note**: Your matrices can be either space-separated or tab-separated.

**Sample Input**:

```
0.289
--------
A B C D E
--------
EBA
E-D
EB-
EED
EBD
EBE
E-D
E-D
```

**Sample Output**:

```
	S	I0	M1	D1	I1	M2	D2	I2	E	
S	0	0	1.0	0	0	0	0	0	0
I0	0	0	0	0	0	0	0	0	0
M1	0	0	0	0	0.625	0.375	0	0	0
D1	0	0	0	0	0	0	0	0	0
I1	0	0	0	0	0	0.8	0.2	0	0
M2	0	0	0	0	0	0	0	0	1.0
D2	0	0	0	0	0	0	0	0	1.0
I2	0	0	0	0	0	0	0	0	0
E	0	0	0	0	0	0	0	0	0
--------
	A	B	C	D	E
S	0	0	0	0	0
I0	0	0	0	0	0
M1	0	0	0	0	1.0
D1	0	0	0	0	0
I1	0	0.8	0	0	0.2
M2	0.143	0	0	0.714	0.143
D2	0	0	0	0	0
I2	0	0	0	0	0
E	0	0	0	0	0
```

In [119]:
def parse_profile_hmm_input(lines):
    lines = [l.strip() for l in lines]
    theta = float(lines[0])
    alphabet = lines[2].split()
    alignment = lines[4:]
    return theta, alphabet, alignment

def parse_profile_hmm_pseudocounts_input(lines):
    lines = [l.strip() for l in lines]
    theta_sigma = lines[0].split()
    theta = float(theta_sigma[0])
    sigma = float(theta_sigma[1])
    alphabet = lines[2].split()
    alignment = lines[4:]
    return theta, sigma, alphabet, alignment

def solve_profile_hmm(theta, alphabet, alignment, sigma=0.0):
    n_rows = len(alignment)
    n_cols = len(alignment[0])
    
    # Determine Match columns vs Insert columns.
    # Columns with a fraction of gaps below theta are conserved (Match states).
    # Columns with too many gaps are variable regions (Insert states).
    match_cols = []
    for j in range(n_cols):
        gaps = sum(1 for row in alignment if row[j] == '-')
        if gaps / n_rows < theta:
            match_cols.append(j)
    
    num_match_states = len(match_cols)
    match_cols_set = set(match_cols)
    
    # Construct state space: S, I0, then M_k, D_k, I_k for each match column, ending with E.
    states = ['S', 'I0']
    for k in range(1, num_match_states + 1):
        states.extend([f'M{k}', f'D{k}', f'I{k}'])
    states.append('E')
    
    transition_counts = {s: {s2: 0.0 for s2 in states} for s in states}
    emission_counts = {s: {a: 0.0 for a in alphabet} for s in states}

    # Helper to identify valid transitions for normalization denominators.
    # Profile HMM structure restricts transitions (e.g., M_i can only go to I_i, M_{i+1}, D_{i+1}).
    def get_allowed_transitions(state):
        allowed = []
        if state == 'S':
            if num_match_states > 0:
                allowed = ['I0', 'M1', 'D1'] 
            else:
                allowed = ['I0', 'E'] 
        elif state == 'E':
            return []
        else:
            kind = state[0]
            idx_str = state[1:]
            idx = int(idx_str)
            
            if kind in ['M', 'D', 'I']:
                # Allow transition to insertion state of current node
                allowed.append(f'I{idx}')
                
                next_idx = idx + 1
                if next_idx <= num_match_states:
                    # Allow transition to next match or deletion state
                    allowed.append(f'M{next_idx}')
                    allowed.append(f'D{next_idx}')
                else:
                    allowed.append('E')
        return allowed
    
    # Iterate over the alignment to count observed state transitions and emissions.
    for row in alignment:
        curr_state = 'S'
        match_idx = 0 
        
        col_idx = 0
        while col_idx < n_cols:
            symbol = row[col_idx]
            
            if col_idx in match_cols_set:
                match_idx += 1
                state_suffix = str(match_idx)
                
                if symbol == '-':
                    # Gap in a match column -> Deletion state (D_j)
                    next_state = f'D{state_suffix}'
                    transition_counts[curr_state][next_state] += 1
                    curr_state = next_state
                else:
                    # Symbol in a match column -> Match state (M_j)
                    next_state = f'M{state_suffix}'
                    transition_counts[curr_state][next_state] += 1
                    emission_counts[next_state][symbol] += 1
                    curr_state = next_state
            else:
                # Column is an insert column.
                # If symbol is present, it's an Insertion state (I_j). Gaps in insert columns are ignored.
                if symbol == '-':
                    pass
                else:
                    next_state = f'I{match_idx}'
                    transition_counts[curr_state][next_state] += 1
                    emission_counts[next_state][symbol] += 1
                    curr_state = next_state
            
            col_idx += 1
            
        transition_counts[curr_state]['E'] += 1
        
    # Apply pseudocounts (Laplace smoothing) to avoid zero probabilities for unobserved events.
    def normalize_counts(counts_dict, possible_targets):
        total = sum(counts_dict.values())
        k = len(possible_targets)
        normalized = {}
        
        # Denominator = total observed counts + total pseudocount mass.
        denominator = (1.0 if total > 0 else 0.0) + k * sigma
        
        if denominator == 0:
            return {t: 0.0 for t in possible_targets}

        for t in possible_targets:
            # Numerator = observed count + pseudocount sigma.
            observed_freq = (counts_dict[t] / total) if total > 0 else 0.0
            prob = (observed_freq + sigma) / denominator
            normalized[t] = prob
            
        return normalized

    # Normalize transitions
    normalized_transitions = {s: {s2: 0.0 for s2 in states} for s in states}
    for s in states:
        if s == 'E': continue
        allowed = get_allowed_transitions(s)
        norm = normalize_counts(transition_counts[s], allowed)
        for target, val in norm.items():
            normalized_transitions[s][target] = val
                
    # Normalize emissions
    normalized_emissions = {s: {a: 0.0 for a in alphabet} for s in states}
    for s in states:
        # Only Match and Insertion states emit symbols; Deletion states are silent.
        if s.startswith('M') or s.startswith('I'):
            norm = normalize_counts(emission_counts[s], alphabet)
            for char, val in norm.items():
                normalized_emissions[s][char] = val
                
    return states, normalized_transitions, normalized_emissions

def print_result(states, transitions, emissions, alphabet):
    # Print Transition Matrix
    print("\t" + "\t".join(states))
    for s in states:
        row_vals = []
        for s2 in states:
            val = transitions[s][s2]
            if val == 0:
                row_vals.append("0") 
            elif val == 1.0:
                row_vals.append("1.0")
            else:
                row_vals.append(f"{val:.3g}")
        print(f"{s}\t" + "\t".join(row_vals))
        
    print("--------")
    
    # Print Emission Matrix
    print("\t" + "\t".join(alphabet))
    for s in states:
        row_vals = []
        for a in alphabet:
            val = emissions[s][a]
            if val == 0:
                row_vals.append("0")
            elif val == 1.0:
                row_vals.append("1.0")
            else:
                row_vals.append(f"{val:.3g}")
        print(f"{s}\t" + "\t".join(row_vals))

In [120]:
### Sample Input Execution
sample_input = """
0.289
--------
A B C D E
--------
EBA
E-D
EB-
EED
EBD
EBE
E-D
E-D
""".strip().split('\n')

theta, alphabet, alignment = parse_profile_hmm_input(sample_input)
states, transitions, emissions = solve_profile_hmm(theta, alphabet, alignment)

print_result(states, transitions, emissions, alphabet)

### Verification
expected_transitions = {
    'S': {'M1': 1.0},
    'M1': {'I1': 0.625, 'M2': 0.375},
    'I1': {'M2': 0.8, 'D2': 0.2},
    'M2': {'E': 1.0},
    'D2': {'E': 1.0},
    'I2': {'E': 0.0} # Checking the zero case
}
expected_emissions = {
    'M1': {'E': 1.0},
    'I1': {'B': 0.8, 'E': 0.2},
    'M2': {'A': 0.143, 'D': 0.714, 'E': 0.143}
}

def verify_output(states, transitions, emissions, expected_transitions, expected_emissions, tolerance=0.01):
    print("Verifying Output...")
    for s in expected_transitions:
        for s2, expected_val in expected_transitions[s].items():
            val = transitions[s][s2]
            assert abs(val - expected_val) <= tolerance, f"Transition {s}->{s2}: Expected {expected_val}, got {val}"
    for s in expected_emissions:
        for a, expected_val in expected_emissions[s].items():
            val = emissions[s][a]
            assert abs(val - expected_val) <= tolerance, f"Emission {s}->{a}: Expected {expected_val}, got {val}"
    print("Sample test passed!")

verify_output(states, transitions, emissions, expected_transitions, expected_emissions)

	S	I0	M1	D1	I1	M2	D2	I2	E
S	0	0	1.0	0	0	0	0	0	0
I0	0	0	0	0	0	0	0	0	0
M1	0	0	0	0	0.625	0.375	0	0	0
D1	0	0	0	0	0	0	0	0	0
I1	0	0	0	0	0	0.8	0.2	0	0
M2	0	0	0	0	0	0	0	0	1.0
D2	0	0	0	0	0	0	0	0	1.0
I2	0	0	0	0	0	0	0	0	0
E	0	0	0	0	0	0	0	0	0
--------
	A	B	C	D	E
S	0	0	0	0	0
I0	0	0	0	0	0
M1	0	0	0	0	1.0
D1	0	0	0	0	0
I1	0	0.8	0	0	0.2
M2	0.143	0	0	0.714	0.143
D2	0	0	0	0	0
I2	0	0	0	0	0
E	0	0	0	0	0
Verifying Output...
Sample test passed!


In [121]:
### Dataset Test
dataset_filename = 'dataset_30331_15.txt' 

try:
    from utils import get_dataset_lines
    lines = get_dataset_lines(dataset_filename)
    if lines:
        theta, alphabet, alignment = parse_profile_hmm_input(lines)
        states, transitions, emissions = solve_profile_hmm(theta, alphabet, alignment)
        print_result(states, transitions, emissions, alphabet)
            
except FileNotFoundError:
    print(f"File {dataset_filename} not found. Please download the dataset.")
except Exception as e:
    print(f"An error occurred: {e}")

	S	I0	M1	D1	I1	M2	D2	I2	M3	D3	I3	M4	D4	I4	M5	D5	I5	E
S	0	0.778	0.222	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
I0	0	0	1.0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
M1	0	0	0	0	0	1.0	0	0	0	0	0	0	0	0	0	0	0	0
D1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
I1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
M2	0	0	0	0	0	0	0	0.778	0.222	0	0	0	0	0	0	0	0	0
D2	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
I2	0	0	0	0	0	0	0	0	1.0	0	0	0	0	0	0	0	0	0
M3	0	0	0	0	0	0	0	0	0	0	0	0.889	0.111	0	0	0	0	0
D3	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
I3	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
M4	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1.0	0	0	0
D4	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1.0	0	0	0
I4	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
M5	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0.778	0.222
D5	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
I5	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0.462	0.538
E	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
--------
	A	B	C	D	E
S	0	0	0	0	0
I0	0	0.143	0.143	0.714	0
M1	0.111	0.111	0.556	0.111	0.111
D1	0	0	0	0	0
I1	0	0	0	0	0
M2	0	0.778	0.111	0	0.111
D2	0	0	0	0	0
I2	0	0	0.857	0	0.143
M3	0	0.111	0.889	0	0
D

# Profile HMM with Pseudocounts Problem

Construct a profile HMM with pseudocounts from a multiple alignment.

**Input**: A multiple alignment $Alignment$, a threshold value $\theta$, and a pseudocount value $\sigma$.

**Output**: $HMM(Alignment, \theta)$, in the form of transition and emission matrices.

**Code Challenge**: Solve the Profile HMM with Pseudocounts Problem.

**Input**: A threshold $\theta$ and a pseudocount $\sigma$, followed by an alphabet $\Sigma$, followed by a multiple alignment $Alignment$ whose strings are formed from $\Sigma$.

**Output**: The transition and emission matrices of $HMM(Alignment, \theta, \sigma)$.

**Note**: Your matrices can be either space-separated or tab-separated.

**Sample Input**:

```
0.358 0.01
--------
A B C D E
--------
A-A
ADA
ACA
A-C
-EA
D-A
```

**Sample Output**:

```
	S	I0	M1	D1	I1	M2	D2	I2	E
S	0	0.01	0.819	0.172	0	0	0	0	0
I0	0	0.333	0.333	0.333	0	0	0	0	0
M1	0	0	0	0	0.398	0.592	0.01	0	0
D1	0	0	0	0	0.981	0.01	0.01	0	0
I1	0	0	0	0	0.01	0.981	0.01	0	0
M2	0	0	0	0	0	0	0	0.01	0.99
D2	0	0	0	0	0	0	0	0.5	0.5
I2	0	0	0	0	0	0	0	0.5	0.5
E	0	0	0	0	0	0	0	0	0
--------
	A	B	C	D	E
S	0	0	0	0	0
I0	0.2	0.2	0.2	0.2	0.2
M1	0.771	0.01	0.01	0.2	0.01
D1	0	0	0	0	0
I1	0.01	0.01	0.327	0.327	0.327
M2	0.803	0.01	0.168	0.01	0.01
D2	0	0	0	0	0
I2	0.2	0.2	0.2	0.2	0.2
E	0	0	0	0	0
```

In [122]:
### Sample Input Execution
sample_input = """
0.358 0.01
--------
A B C D E
--------
A-A
ADA
ACA
A-C
-EA
D-A
""".strip().split('\n')

theta, sigma, alphabet, alignment = parse_profile_hmm_pseudocounts_input(sample_input)
states, transitions, emissions = solve_profile_hmm(theta, alphabet, alignment, sigma=sigma)

print_result(states, transitions, emissions, alphabet)

### Verification
# Selected values from expected output
expected_transitions = {
    'S': {'I0': 0.01, 'M1': 0.819, 'D1': 0.172},
    'M2': {'E': 0.99, 'I2': 0.01},
    'I0': {'I0': 0.333, 'M1': 0.333}
}
expected_emissions = {
    'M1': {'A': 0.771, 'D': 0.2, 'E': 0.01},
    'I1': {'C': 0.327},
    'I0': {'A': 0.2}
}

def verify_output(states, transitions, emissions, expected_transitions, expected_emissions, tolerance=0.01):
    for s in expected_transitions:
        for s2, expected_val in expected_transitions[s].items():
            val = transitions[s][s2]
            assert abs(val - expected_val) <= tolerance, f"Transition {s}->{s2}: Expected {expected_val}, got {val}"
    for s in expected_emissions:
        for a, expected_val in expected_emissions[s].items():
            val = emissions[s][a]
            assert abs(val - expected_val) <= tolerance, f"Emission {s}->{a}: Expected {expected_val}, got {val}"

verify_output(states, transitions, emissions, expected_transitions, expected_emissions)
print("Sample test passed!")

	S	I0	M1	D1	I1	M2	D2	I2	E
S	0	0.00971	0.819	0.172	0	0	0	0	0
I0	0	0.333	0.333	0.333	0	0	0	0	0
M1	0	0	0	0	0.398	0.592	0.00971	0	0
D1	0	0	0	0	0.981	0.00971	0.00971	0	0
I1	0	0	0	0	0.00971	0.981	0.00971	0	0
M2	0	0	0	0	0	0	0	0.0098	0.99
D2	0	0	0	0	0	0	0	0.5	0.5
I2	0	0	0	0	0	0	0	0.5	0.5
E	0	0	0	0	0	0	0	0	0
--------
	A	B	C	D	E
S	0	0	0	0	0
I0	0.2	0.2	0.2	0.2	0.2
M1	0.771	0.00952	0.00952	0.2	0.00952
D1	0	0	0	0	0
I1	0.00952	0.00952	0.327	0.327	0.327
M2	0.803	0.00952	0.168	0.00952	0.00952
D2	0	0	0	0	0
I2	0.2	0.2	0.2	0.2	0.2
E	0	0	0	0	0
Sample test passed!


In [123]:
### Dataset Test
dataset_filename = 'dataset_30332_5.txt' 

try:
    from utils import get_dataset_lines
    lines = get_dataset_lines(dataset_filename)
    if lines:
        theta, sigma, alphabet, alignment = parse_profile_hmm_pseudocounts_input(lines)
        states, transitions, emissions = solve_profile_hmm(theta, alphabet, alignment, sigma=sigma)
        print_result(states, transitions, emissions, alphabet)
            
except FileNotFoundError:
    print(f"File {dataset_filename} not found. Please download the dataset.")
except Exception as e:
    print(f"An error occurred: {e}")

	S	I0	M1	D1	I1	M2	D2	I2	M3	D3	I3	E
S	0	0.00971	0.842	0.148	0	0	0	0	0	0	0	0
I0	0	0.333	0.333	0.333	0	0	0	0	0	0	0	0
M1	0	0	0	0	0.00971	0.981	0.00971	0	0	0	0	0
D1	0	0	0	0	0.00971	0.981	0.00971	0	0	0	0	0
I1	0	0	0	0	0.333	0.333	0.333	0	0	0	0	0
M2	0	0	0	0	0	0	0	0.00971	0.842	0.148	0	0
D2	0	0	0	0	0	0	0	0.333	0.333	0.333	0	0
I2	0	0	0	0	0	0	0	0.333	0.333	0.333	0	0
M3	0	0	0	0	0	0	0	0	0	0	0.0098	0.99
D3	0	0	0	0	0	0	0	0	0	0	0.0098	0.99
I3	0	0	0	0	0	0	0	0	0	0	0.5	0.5
E	0	0	0	0	0	0	0	0	0	0	0	0
--------
	A	B	C	D	E
S	0	0	0	0	0
I0	0.2	0.2	0.2	0.2	0.2
M1	0.00952	0.962	0.00952	0.00952	0.00952
D1	0	0	0	0	0
I1	0.2	0.2	0.2	0.2	0.2
M2	0.146	0.00952	0.00952	0.69	0.146
D2	0	0	0	0	0
I2	0.2	0.2	0.2	0.2	0.2
M3	0.962	0.00952	0.00952	0.00952	0.00952
D3	0	0	0	0	0
I3	0.2	0.2	0.2	0.2	0.2
E	0	0	0	0	0


# Sequence Alignment with Profile HMM Problem

Align a new sequence to a family of sequences using a profile HMM.

**Input**: A multiple alignment $Alignment$, a threshold $\theta$, a pseudocount value $\sigma$, and a string $Text$.

**Output**: An optimal hidden path emitting $Text$ in $HMM(Alignment, \theta, \sigma)$.

**Code Challenge**: Solve the Sequence Alignment with Profile HMM Problem.

**Input**: A string $x$ followed by a threshold $\theta$ and a pseudocount $\sigma$, followed by an alphabet $\Sigma$, followed by a multiple alignment $Alignment$ whose strings are formed from $\Sigma$.

**Output**: An optimal hidden path emitting $x$ in $HMM(Alignment, \theta, \sigma)$.

**Sample Input**:

```
AEFDFDC
--------
0.4 0.01
--------
A B C D E F
--------
ACDEFACADF
AFDA---CCF
A--EFD-FDC
ACAEF--A-C
ADDEFAAADF
```

**Sample Output**:

```
M1 D2 D3 M4 M5 I5 M6 M7 M8
```

In [124]:
def parse_sequence_alignment_input(lines):
    lines = [l.strip() for l in lines]
    text = lines[0]
    # Reuse existing HMM parser starting from the second line.
    theta, sigma, alphabet, alignment = parse_profile_hmm_pseudocounts_input(lines[2:])
    return text, theta, sigma, alphabet, alignment

def solve_alignment_profile_hmm(text, states, transitions, emissions):
    # Construct reverse graph (adjacency list of incoming edges) to enable efficient DP lookups.
    parents = {s: [] for s in states}
    for u in states:
        for v, prob in transitions[u].items():
            if prob > 0:
                parents[v].append(u)
    
    n = len(text)
    # V[i][state] stores the max probability of the optimal path emitting text[:i] and ending in `state`.
    V = [{s: 0.0 for s in states} for _ in range(n + 1)]
    # Backtrack matrix stores the previous state that maximized the probability for reconstruction.
    Backtrack = [{s: None for s in states} for _ in range(n + 1)]
    
    V[0]['S'] = 1.0
    
    # Process states column by column (i from 0 to n).
    # Within each column, we must handle Silent states (Deletions) carefully.
    # Silent states do not emit a symbol, so they stay in column i (depending on V[i]).
    # Emitting states consume a symbol, so they depend on V[i-1].
    
    # Initialize Silent states reachable from S at i=0 (e.g., D1, D2 chain).
    # Since Deletion states in Profile HMM form a directed acyclic chain (D1->D2...), 
    # we can process them in topological order effectively by using the standard state list order.
    for s in states:
        if s == 'S': continue
        if s.startswith('D'):
            max_p = 0.0
            best_prev = None
            for prev in parents[s]:
                p = V[0][prev] * transitions[prev][s]
                if p > max_p:
                    max_p = p
                    best_prev = prev
            
            V[0][s] = max_p
            Backtrack[0][s] = best_prev

    for i in range(1, n + 1):
        char = text[i-1]
        
        # 1. Update Emitting States (Match, Insertion).
        # These transition from a state in the previous column (i-1).
        for s in states:
            if s.startswith('M') or s.startswith('I'):
                max_p = 0.0
                best_prev = None
                emit_prob = emissions[s].get(char, 0.0)
                
                if emit_prob > 0:
                    for prev in parents[s]:
                        p = V[i-1][prev] * transitions[prev][s]
                        if p > max_p:
                            max_p = p
                            best_prev = prev
                    
                    V[i][s] = max_p * emit_prob
                    Backtrack[i][s] = best_prev
        
        # 2. Update Silent States (Deletion).
        # These transition from a state in the *current* column (i), because they consume no text.
        # Iterating in the predefined state order ensures we visit Match/Insert before Deletion (or previous Deletion).
        for s in states:
            if s.startswith('D'):
                max_p = 0.0
                best_prev = None
                for prev in parents[s]:
                    p = V[i][prev] * transitions[prev][s]
                    if p > max_p:
                        max_p = p
                        best_prev = prev
                
                V[i][s] = max_p
                Backtrack[i][s] = best_prev
                
    # Calculate probability of transitioning to the End state (E) from the last column n.
    max_p = 0.0
    best_prev = None
    target = 'E'
    for prev in parents[target]:
        p = V[n][prev] * transitions[prev][target]
        if p > max_p:
            max_p = p
            best_prev = prev
    
    # Backtrack from the state that best transitions to End.
    path = []
    curr_state = best_prev 
    curr_i = n
    
    while curr_state and curr_state != 'S':
        path.append(curr_state)
        
        prev_state = Backtrack[curr_i][curr_state]
        
        # If current state emitted a character, we step back in the text index i.
        is_emitting = curr_state.startswith('M') or curr_state.startswith('I')
        if is_emitting:
            curr_i -= 1
            
        curr_state = prev_state
        
    path.reverse()
    return path

In [125]:
### Sample Input Execution
sample_input = """
AEFDFDC
--------
0.4 0.01
--------
A B C D E F
--------
ACDEFACADF
AFDA---CCF
A--EFD-FDC
ACAEF--A-C
ADDEFAAADF
""".strip().split('\n')

text, theta, sigma, alphabet, alignment = parse_sequence_alignment_input(sample_input)
states, transitions, emissions = solve_profile_hmm(theta, alphabet, alignment, sigma=sigma)
path = solve_alignment_profile_hmm(text, states, transitions, emissions)

print(" ".join(path))

### Verification
expected_path = "M1 D2 D3 M4 M5 I5 M6 M7 M8".split()
assert path == expected_path, f"Expected {expected_path}, got {path}"
print("Sample test passed!")

M1 D2 D3 M4 M5 I5 M6 M7 M8
Sample test passed!


In [126]:
### Dataset Test
dataset_filename = 'dataset_30332_14.txt' 

try:
    lines = get_dataset_lines(dataset_filename)
    if lines:
        text, theta, sigma, alphabet, alignment = parse_sequence_alignment_input(lines)
        states, transitions, emissions = solve_profile_hmm(theta, alphabet, alignment, sigma=sigma)
        path = solve_alignment_profile_hmm(text, states, transitions, emissions)
        print(" ".join(path))
            
except FileNotFoundError:
    print(f"File {dataset_filename} not found. Please download the dataset.")
except Exception as e:
    print(f"An error occurred: {e}")

M1 M2 M3 M4 M5 M6 M7 M8 M9 M10 M11 M12 I12 I12 D13 I13 M14 M15 M16 M17 M18 M19 M20 D21 D22 M23 D24 M25 M26 M27 I27 I27 I27 M28 M29 M30 I30 M31 I31 I31 D32 M33 M34 D35 I35 M36 M37 I37 M38 M39 M40 M41 M42 I42 M43


# HMM Parameter Estimation Problem

**Code Challenge**: Solve the HMM Parameter Estimation Problem.

**Input**: A string $x$ of symbols emitted from an HMM, followed by the HMM's alphabet $\Sigma$, followed by a path $\pi$, followed by the collection of states of the HMM.

**Output**: A transition matrix $Transition$ followed by an emission matrix $Emission$ that maximize $Pr(x, \pi)$ over all possible transition and emission matrices.

**Sample Input**:

```
yzzzyxzxxx
--------
x y z
--------
BBABABABAB
--------
A B C
```

**Sample Output**:

```
	A	B	C
A	0.0	1.0	0.0
B	0.8	0.2	0.0
C	0.333	0.333	0.333
--------
	x	y	z
A	0.25	0.25	0.5
B	0.5	0.167	0.333
C	0.333	0.333	0.333
```

In [127]:
def parse_hmm_parameter_estimation_input(lines):
    # Filter out separator lines
    lines = [l.strip() for l in lines if l.strip() != '--------']
    x = lines[0]
    alphabet = lines[1].split()
    path = lines[2]
    states = lines[3].split()
    return x, alphabet, path, states

def solve_hmm_parameter_estimation(x, alphabet, path, states):
    """
    Estimates HMM parameters (transitions and emissions) using Maximum Likelihood Estimation (MLE).
    Given the observed path and emitted string, we count the frequencies of transitions and emissions.
    """
    n_states = len(states)
    n_alphabet = len(alphabet)
    
    # Initialize count matrices.
    transitions = {s: {s2: 0.0 for s2 in states} for s in states}
    emissions = {s: {a: 0.0 for a in alphabet} for s in states}
    
    # Calculate transition counts (how often state u transitions to v).
    for i in range(len(path) - 1):
        curr_s = path[i]
        next_s = path[i+1]
        transitions[curr_s][next_s] += 1.0
        
    # Calculate emission counts (how often state u emits symbol a).
    for i in range(len(x)):
        curr_s = path[i]
        symbol = x[i]
        emissions[curr_s][symbol] += 1.0
        
    # Normalize Transitions to get probabilities.
    # P(v|u) = count(u->v) / sum(count(u->k) for all k)
    for s in states:
        total = sum(transitions[s].values())
        if total > 0:
            for s2 in states:
                transitions[s][s2] /= total
        else:
            # Handle edge case where a state is never visited or has no outgoing transitions:
            # Assign uniform probability distribution.
            prob = 1.0 / n_states
            for s2 in states:
                transitions[s][s2] = prob
                
    # Normalize Emissions to get probabilities.
    # P(a|u) = count(u emits a) / sum(count(u emits k) for all k)
    for s in states:
        total = sum(emissions[s].values())
        if total > 0:
            for a in alphabet:
                emissions[s][a] /= total
        else:
            # Handle edge case where a state never emits:
            # Assign uniform probability distribution.
            prob = 1.0 / n_alphabet
            for a in alphabet:
                emissions[s][a] = prob
                
    return transitions, emissions

def print_hmm_parameters(states, transitions, emissions, alphabet):
    # Print Transition Matrix
    print("\t" + "\t".join(states))
    for s in states:
        row_vals = []
        for s2 in states:
            val = transitions[s][s2]
            if val == 0:
                row_vals.append("0") 
            elif val == 1.0:
                row_vals.append("1.0")
            else:
                row_vals.append(f"{val:.3g}")
        print(f"{s}\t" + "\t".join(row_vals))
        
    print("--------")
    
    # Print Emission Matrix
    print("\t" + "\t".join(alphabet))
    for s in states:
        row_vals = []
        for a in alphabet:
            val = emissions[s][a]
            if val == 0:
                row_vals.append("0")
            elif val == 1.0:
                row_vals.append("1.0")
            else:
                row_vals.append(f"{val:.3g}")
        print(f"{s}\t" + "\t".join(row_vals))

In [ ]:
### Sample Input Execution
sample_input = """
yzzzyxzxxx
--------
x y z
--------
BBABABABAB
--------
A B C
""".strip().split('\n')

seq, alphabet, path, states = parse_hmm_parameter_estimation_input(sample_input)
transitions, emissions = solve_hmm_parameter_estimation(seq, alphabet, path, states)

print_hmm_parameters(states, transitions, emissions, alphabet)

### Verification
expected_transitions = {
    'A': {'B': 1.0, 'A': 0.0, 'C': 0.0},
    'B': {'A': 0.8, 'B': 0.2, 'C': 0.0},
    'C': {'A': 0.333, 'B': 0.333, 'C': 0.333}
}
expected_emissions = {
    'A': {'x': 0.25, 'y': 0.25, 'z': 0.5},
    'B': {'x': 0.5, 'y': 0.167, 'z': 0.333},
    'C': {'x': 0.333, 'y': 0.333, 'z': 0.333}
}

def verify_hmm_parameters(transitions, emissions, expected_transitions, expected_emissions, tolerance=0.01):
    print("\nVerifying Output...")
    for s in expected_transitions:
        for s2, expected_val in expected_transitions[s].items():
            val = transitions[s][s2]
            # Use approximately equal check for floats
            assert abs(val - expected_val) <= tolerance, f"Transition {s}->{s2}: Expected {expected_val}, got {val}"
    for s in expected_emissions:
        for a, expected_val in expected_emissions[s].items():
            val = emissions[s][a]
            assert abs(val - expected_val) <= tolerance, f"Emission {s}->{a}: Expected {expected_val}, got {val}"
        
    print("Sample test passed!")

verify_hmm_parameters(transitions, emissions, expected_transitions, expected_emissions)

	A	B	C
A	0	1.0	0
B	0.8	0.2	0
C	0.333	0.333	0.333
--------
	x	y	z
A	0.25	0.25	0.5
B	0.5	0.167	0.333
C	0.333	0.333	0.333

Verifying Output...
Sample test passed!


In [ ]:
### Dataset Test
dataset_filename = 'dataset_30333_4.txt'

try:
    from utils import get_dataset_lines
    lines = get_dataset_lines(dataset_filename)
    if lines:
        seq, alphabet, path, states = parse_hmm_parameter_estimation_input(lines)
        transitions, emissions = solve_hmm_parameter_estimation(seq, alphabet, path, states)
        print_hmm_parameters(states, transitions, emissions, alphabet)
            
except FileNotFoundError:
    print(f"File {dataset_filename} not found. Please download the dataset.")
except Exception as e:
    print(f"An error occurred: {e}")

	A	B
A	0.472	0.528
B	0.609	0.391
--------
	x	y	z
A	0.333	0.241	0.426
B	0.37	0.283	0.348


# Viterbi Learning

**Code Challenge**: Implement Viterbi learning for estimating the parameters of an HMM.

**Input**: A number of iterations $j$, followed by a string $x$ of symbols emitted by an HMM, followed by the HMM's alphabet $\Sigma$, followed by the HMM's states, followed by initial transition and emission matrices for the HMM.

**Output**: Emission and transition matrices resulting from applying Viterbi learning for $j$ iterations.

**Sample Input**:

```
100
--------
zyzxzxxxzz
--------
x y z
--------
A B
--------
	A	B
A	0.599	0.401	
B	0.294	0.706	
--------
	x	y	z
A	0.424	0.367	0.209	
B	0.262	0.449	0.289
```

**Sample Output**:

```
	A	B
A	0.5	0.5	
B	0.0	1.0	
--------
	x	y	z
A	0.333	0.333	0.333	
B	0.4	0.1	0.5
```

In [130]:
def parse_matrix(lines, row_labels_expected, col_labels_expected):
    """Parses a matrix with header row and header column."""
    col_headers = lines[0].split()
    matrix = {}
    
    for i in range(1, len(lines)):
        parts = lines[i].split()
        row_label = parts[0]
        vals = [float(v) for v in parts[1:]]
        
        matrix[row_label] = {}
        for j, val in enumerate(vals):
            col_label = col_headers[j]
            matrix[row_label][col_label] = val
            
    return matrix

def parse_viterbi_learning_input(lines):
    # Split input sections by the delimiter line.
    parts = []
    current_part = []
    for line in lines:
        if line.strip() == '--------':
            if current_part:
                parts.append(current_part)
            current_part = []
        else:
            current_part.append(line.strip())
    if current_part:
        parts.append(current_part)
        
    iterations = int(parts[0][0])
    x = parts[1][0]
    alphabet = parts[2][0].split()
    states = parts[3][0].split()
    
    transitions = parse_matrix(parts[4], states, states)
    emissions = parse_matrix(parts[5], states, alphabet)
    
    return iterations, x, alphabet, states, transitions, emissions

def viterbi_algorithm(x, states, transitions, emissions):
    n = len(x)
    T = len(states)
    
    # Initialize DP tables.
    # scores[i][state] holds max log-probability (or probability) of path ending at state at step i.
    scores = [{s: 0.0 for s in states} for _ in range(n)]
    backtrack = [{s: None for s in states} for _ in range(n)]
    
    # Initialization Step (i=0):
    # Probability = Prior(Start) * Emission(Start)
    # We assume uniform prior 1/|States| if not given.
    start_prob = 1.0 / len(states)
    for s in states:
        scores[0][s] = start_prob * emissions[s][x[0]]
        
    # Recursion Step (i=1 to n-1):
    # Maximize (Previous Score * Transition * Emission)
    for i in range(1, n):
        symbol = x[i]
        for s in states:
            max_score = -1.0
            best_prev = None
            
            emit_prob = emissions[s][symbol]
            
            for prev_s in states:
                # Calculate probability of arriving at s from prev_s
                score = scores[i-1][prev_s] * transitions[prev_s][s] * emit_prob
                
                if score > max_score:
                    max_score = score
                    best_prev = prev_s
            
            scores[i][s] = max_score
            backtrack[i][s] = best_prev
            
    # Termination Step: Find the state with the highest score at the final step.
    max_final_score = -1.0
    last_state = None
    for s in states:
        if scores[n-1][s] > max_final_score:
            max_final_score = scores[n-1][s]
            last_state = s
            
    # Backtracking: Reconstruct the optimal path by following pointers backward.
    path = []
    curr_state = last_state
    for i in range(n-1, -1, -1):
        path.append(curr_state)
        curr_state = backtrack[i][curr_state]
        
    return "".join(path[::-1])

def viterbi_learning(iterations, x, alphabet, states, transitions, emissions):
    """
    Viterbi Learning estimates HMM parameters when the path is unknown.
    It iteratively maximizes the parameters based on the most likely path (Viterbi path).
    """
    current_transitions = transitions
    current_emissions = emissions
    
    for _ in range(iterations):
        # E-Step (Classification): Find the most likely path pi* for the current parameters.
        path = viterbi_algorithm(x, states, current_transitions, current_emissions)
        
        # M-Step (Maximization): Update parameters to maximize Likelihood(x, pi*)
        # using the standard parameter estimation (counting).
        current_transitions, current_emissions = solve_hmm_parameter_estimation(x, alphabet, path, states)
        
    return current_transitions, current_emissions

In [ ]:
### Sample Input Execution
sample_input = """
100
--------
zyzxzxxxzz
--------
x y z
--------
A B
--------
	A	B
A	0.599	0.401
B	0.294	0.706
--------
	x	y	z
A	0.424	0.367	0.209
B	0.262	0.449	0.289
""".strip().split('\n')

iterations, seq, alphabet, states, transitions, emissions = parse_viterbi_learning_input(sample_input)
final_transitions, final_emissions = viterbi_learning(iterations, seq, alphabet, states, transitions, emissions)

print_hmm_parameters(states, final_transitions, final_emissions, alphabet)

### Verification
# Check against sample output from problem description
expected_transitions = {
    'A': {'A': 0.5, 'B': 0.5},
    'B': {'A': 0.0, 'B': 1.0}
}
expected_emissions = {
    'A': {'x': 0.333, 'y': 0.333, 'z': 0.333},
    'B': {'x': 0.4, 'y': 0.1, 'z': 0.5}
}

verify_hmm_parameters(final_transitions, final_emissions, expected_transitions, expected_emissions, tolerance=0.01)

	A	B
A	0.5	0.5
B	0	1.0
--------
	x	y	z
A	0.333	0.333	0.333
B	0.4	0.1	0.5

Verifying Output...
Sample test passed!


In [ ]:
### Dataset Test
dataset_filename = 'dataset_30333_8.txt'

try:
    from utils import get_dataset_lines
    lines = get_dataset_lines(dataset_filename)
    if lines:
        iterations, seq, alphabet, states, transitions, emissions = parse_viterbi_learning_input(lines)
        final_transitions, final_emissions = viterbi_learning(iterations, seq, alphabet, states, transitions, emissions)
        print_hmm_parameters(states, final_transitions, final_emissions, alphabet)
            
except FileNotFoundError:
    print(f"File {dataset_filename} not found. Please download the dataset.")
except Exception as e:
    print(f"An error occurred: {e}")
    import traceback
    traceback.print_exc()

	A	B	C
A	0.333	0.333	0.333
B	0	0	1.0
C	0	0.623	0.377
--------
	x	y	z
A	0.333	0.333	0.333
B	0.667	0.333	0
C	0.18	0.262	0.557


# Soft Decoding Problem

**Code Challenge**: Solve the Soft Decoding Problem.

**Input**: A string $x$, followed by the alphabet $\Sigma$ from which $x$ was constructed, followed by the states $States$, transition matrix $Transition$, and emission matrix $Emission$ of an HMM $(\Sigma, States, Transition, Emission)$.

**Output**: An $|x| \times |States|$ matrix whose $(i, k)$-th element holds the conditional probability $Pr(\pi_i = k|x)$.

**Sample Input**:

```
zyxxxxyxzz
--------
x y z
--------
A B
--------
	A	B
A	0.911	0.089
B	0.228	0.772
--------
	x	y	z
A	0.356	0.191	0.453 
B	0.040	0.467	0.493
```

**Sample Output**:

```
A	B 
0.5438	0.4562 
0.6492	0.3508 
0.9647	0.0353 
0.9936	0.0064 
0.9957	0.0043 
0.9891	0.0109 
0.9154	0.0846 
0.964	0.036 
0.8737	0.1263 
0.8167	0.1833
```

In [133]:
def parse_soft_decoding_input(lines):
    # Split input sections by delimiter.
    parts = []
    current_part = []
    for line in lines:
        if line.strip() == '--------':
            if current_part:
                parts.append(current_part)
            current_part = []
        else:
            current_part.append(line.strip())
    if current_part:
        parts.append(current_part)
        
    x = parts[0][0]
    alphabet = parts[1][0].split()
    states = parts[2][0].split()
    
    # Reuse parse_matrix to load initial HMM parameters.
    transitions = parse_matrix(parts[3], states, states)
    emissions = parse_matrix(parts[4], states, alphabet)
    
    return x, alphabet, states, transitions, emissions

def forward_pass(x, states, transitions, emissions):
    """
    Computes Forward probabilities: alpha_i(state) = P(x_1...x_i, pi_i = state).
    """
    n = len(x)
    fwd = [{s: 0.0 for s in states} for _ in range(n)]
    
    # Initialization (i=0)
    start_prob = 1.0 / len(states)
    for s in states:
        fwd[0][s] = start_prob * emissions[s][x[0]]
        
    # Recursion (i=1 to n-1)
    for i in range(1, n):
        symbol = x[i]
        for s in states:
            # Sum over all previous states k: alpha_{i-1}(k) * trans(k->s)
            sum_prev = sum(fwd[i-1][prev_s] * transitions[prev_s][s] for prev_s in states)
            fwd[i][s] = sum_prev * emissions[s][symbol]
    return fwd

def backward_pass(x, states, transitions, emissions):
    """
    Computes Backward probabilities: beta_i(state) = P(x_{i+1}...x_n | pi_i = state).
    """
    n = len(x)
    bwd = [{s: 0.0 for s in states} for _ in range(n)]
    
    # Initialization (i=n-1): Base case is 1.0.
    for s in states:
        bwd[n-1][s] = 1.0
        
    # Recursion (i=n-2 down to 0)
    for i in range(n - 2, -1, -1):
        next_symbol = x[i+1]
        for s in states:
            # Sum over all next states l: trans(s->l) * emit(l->next_symbol) * beta_{i+1}(l)
            sum_next = sum(bwd[i+1][next_s] * transitions[s][next_s] * emissions[next_s][next_symbol] for next_s in states)
            bwd[i][s] = sum_next
    return bwd

def solve_soft_decoding(x, states, transitions, emissions):
    """
    Calculates posterior probabilities P(pi_i = k | x) for all i, k.
    Uses Forward-Backward algorithm.
    """
    fwd = forward_pass(x, states, transitions, emissions)
    bwd = backward_pass(x, states, transitions, emissions)
    
    n = len(x)
    probs = [{s: 0.0 for s in states} for _ in range(n)]
    
    # Total probability P(x) is sum of forward probabilities at the last step.
    p_x = sum(fwd[n-1].values())
    
    for i in range(n):
        for s in states:
            if p_x == 0:
                probs[i][s] = 0.0
            else:
                # P(pi_i = k | x) = (alpha_i(k) * beta_i(k)) / P(x)
                probs[i][s] = (fwd[i][s] * bwd[i][s]) / p_x
            
    return probs

def print_soft_decoding(states, probs):
    print("\t".join(states))
    for row in probs:
        vals = [f"{row[s]:.4g}" for s in states] 
        print("\t".join(vals))

In [ ]:
### Sample Input Execution
sample_input = """
zyxxxxyxzz
--------
x y z
--------
A B
--------
	A	B
A	0.911	0.089
B	0.228	0.772
--------
	x	y	z
A	0.356	0.191	0.453 
B	0.040	0.467	0.493
""".strip().split('\n')

seq, alphabet, states, transitions, emissions = parse_soft_decoding_input(sample_input)
probs = solve_soft_decoding(seq, states, transitions, emissions)

print_soft_decoding(states, probs)

### Verification
# Validate first and last rows roughly against sample output
expected_first_row = {'A': 0.5438, 'B': 0.4562}
expected_last_row = {'A': 0.8167, 'B': 0.1833}

def verify_soft_decoding(probs, expected_row, row_idx, tolerance=0.001):
    print("\nVerifying row", row_idx, "...")
    for s, val in expected_row.items():
        calc_val = probs[row_idx][s]
        assert abs(calc_val - val) <= tolerance, f"State {s}: Expected {val}, got {calc_val}"
    print("Passed.")

verify_soft_decoding(probs, expected_first_row, 0)
verify_soft_decoding(probs, expected_last_row, -1)
print("Sample test passed!")

A	B
0.5438	0.4562
0.6492	0.3508
0.9647	0.03532
0.9936	0.006376
0.9957	0.004334
0.9891	0.01087
0.9154	0.08462
0.964	0.03597
0.8737	0.1263
0.8167	0.1833

Verifying row 0 ...
Passed.

Verifying row -1 ...
Passed.
Sample test passed!


In [ ]:
### Dataset Test
dataset_filename = 'dataset_30334_5.txt'

try:
    lines = get_dataset_lines(dataset_filename)
    if lines:
        seq, alphabet, states, transitions, emissions = parse_soft_decoding_input(lines)
        probs = solve_soft_decoding(seq, states, transitions, emissions)
        print_soft_decoding(states, probs)
            
except FileNotFoundError:
    print(f"File {dataset_filename} not found. Please download the dataset.")
except Exception as e:
    print(f"An error occurred: {e}")
    import traceback
    traceback.print_exc()

A	B
0.4343	0.5657
0.4144	0.5856
0.4666	0.5334
0.661	0.339
0.6874	0.3126
0.6846	0.3154
0.5533	0.4467
0.6677	0.3323
0.7219	0.2781
0.7084	0.2916


# Baum-Welch Learning

**Code Challenge**: Implement Baum-Welch Learning.

**Input**: A sequence of emitted symbols $x = x_1 \dots x_n$ in an alphabet $A$, generated by a $k$-state HMM with unknown transition and emission probabilities, initial $Transition$ and $Emission$ matrices, and a number of iterations $j$.

**Output**: A matrix of transition probabilities $Transition$ and a matrix of emission probabilities $Emission$ that maximizes $Pr(x, \pi)$ over all possible transition and emission matrices and over all hidden paths $\pi$.

**Sample Input**:

```
10
--------
xzyyzyzyxy
--------
x	y	z
--------
A	B
--------
	A	B
A	0.019	0.981 
B	0.668	0.332 
--------
x	y	z
A	0.175	0.003	0.821 
B	0.196	0.512	0.293
```

**Sample Output**:

```
	A	B
A	0.000	1.000	
B	0.786	0.214	
--------
	x	y	z
A	0.242	0.000	0.758	
B	0.172	0.828	0.000
```

In [136]:
def solve_baum_welch(iterations, x, alphabet, states, transitions, emissions):
    """
    Baum-Welch Learning implements the Expectation-Maximization (EM) algorithm for HMMs.
    It iteratively estimates parameters when path is unknown by using soft counts (expectations).
    """
    for _ in range(iterations):
        # E-Step: Calculate expected state occupancies and transitions using Forward-Backward.
        fwd = forward_pass(x, states, transitions, emissions)
        bwd = backward_pass(x, states, transitions, emissions)
        
        n = len(x)
        p_x = sum(fwd[n-1].values())
        
        # Accumulators for expected counts (Soft Counts).
        exp_trans_counts = {u: {v: 0.0 for v in states} for u in states}
        exp_emit_counts = {u: {a: 0.0 for a in alphabet} for u in states}
        
        # Calculate expected transition counts: Expected number of times u -> v happens.
        # P(pi_i=u, pi_{i+1}=v | x) = (alpha_i(u) * a_{uv} * b_v(x_{i+1}) * beta_{i+1}(v)) / P(x)
        if p_x != 0:
            for i in range(n - 1):
                char_next = x[i+1]
                for u in states:
                    for v in states:
                        numer = fwd[i][u] * transitions[u][v] * emissions[v][char_next] * bwd[i+1][v]
                        exp_trans_counts[u][v] += numer / p_x
                        
            # Calculate expected emission counts: Expected number of times u emits symbol.
            # P(pi_i=u | x) = (alpha_i(u) * beta_i(u)) / P(x)
            for i in range(n):
                char = x[i]
                for u in states:
                    numer = fwd[i][u] * bwd[i][u]
                    exp_emit_counts[u][char] += numer / p_x
        
        # M-Step: Maximize parameters by normalizing the expected counts.
        
        # Update Transitions: a'_{uv} = ExpCount(u->v) / Sum_k ExpCount(u->k)
        for u in states:
            total_trans = sum(exp_trans_counts[u].values())
            if total_trans > 0:
                for v in states:
                    transitions[u][v] = exp_trans_counts[u][v] / total_trans
            else:
                # If state effectively unvisited/no transitions, distribute uniformly 
                for v in states:
                    transitions[u][v] = 1.0 / len(states)

        # Update Emissions: e'_{u}(a) = ExpCount(u emits a) / Sum_b ExpCount(u emits b)
        for u in states:
            total_emit = sum(exp_emit_counts[u].values())
            if total_emit > 0:
                for a in alphabet:
                    emissions[u][a] = exp_emit_counts[u][a] / total_emit
            else:
                for a in alphabet:
                    emissions[u][a] = 1.0 / len(alphabet)
                    
    return transitions, emissions

In [ ]:
### Sample Input Execution
sample_input = """
10
--------
xzyyzyzyxy
--------
x	y	z
--------
A	B
--------
	A	B
A	0.019	0.981 
B	0.668	0.332 
--------
x	y	z
A	0.175	0.003	0.821 
B	0.196	0.512	0.293
""".strip().split('\n')

# Reuse parser for Viterbi Learning as formatting is identical
iterations, seq, alphabet, states, transitions, emissions = parse_viterbi_learning_input(sample_input)
final_transitions, final_emissions = solve_baum_welch(iterations, seq, alphabet, states, transitions, emissions)

print_hmm_parameters(states, final_transitions, final_emissions, alphabet)

### Verification
expected_transitions = {
    'A': {'A': 0.000, 'B': 1.000},
    'B': {'A': 0.786, 'B': 0.214}
}
expected_emissions = {
    'A': {'x': 0.242, 'y': 0.000, 'z': 0.758},
    'B': {'x': 0.172, 'y': 0.828, 'z': 0.000}
}

verify_hmm_parameters(final_transitions, final_emissions, expected_transitions, expected_emissions, tolerance=0.001)

	A	B
A	6.75e-06	1
B	0.786	0.214
--------
	x	y	z
A	0.242	3.42e-14	0.758
B	0.172	0.828	1e-09

Verifying Output...
Sample test passed!


In [ ]:
### Dataset Test
dataset_filename = 'dataset_30335_5.txt'

try:
    from utils import get_dataset_lines
    lines = get_dataset_lines(dataset_filename)
    if lines:
        iterations, seq, alphabet, states, transitions, emissions = parse_viterbi_learning_input(lines)
        final_transitions, final_emissions = solve_baum_welch(iterations, seq, alphabet, states, transitions, emissions)
        print_hmm_parameters(states, final_transitions, final_emissions, alphabet)
            
except FileNotFoundError:
    print(f"File {dataset_filename} not found. Please download the dataset.")
except Exception as e:
    print(f"An error occurred: {e}")
    import traceback
    traceback.print_exc()

	A	B	C
A	0.00202	0.000755	0.997
B	0.819	0.18	0.000972
C	0.688	0.277	0.0344
--------
	x	y	z
A	0.309	0.00173	0.69
B	1.44e-10	0.984	0.0164
C	0.258	0.44	0.302


In [ ]:
# Coursera Quiz Questions (Module 4-5)

# Q1 Say that we are in a crooked casino where the probability of heads for the biased coin is 3/4, and we observe the following sequence of flips: HTTHH
# Determine whether a fair coin or a biased coin was more likely to have generated this sequence.
def q1_solver():
    # P(Heads|Biased) = 0.75, P(Heads|Fair) = 0.5
    seq = "HTTHH"
    p_b = 0.75
    p_f = 0.5
    
    # Calculate P(seq|Biased)
    prob_biased = 1.0
    for char in seq:
        if char == 'H':
            prob_biased *= p_b
        else:
            prob_biased *= (1 - p_b)
            
    # Calculate P(seq|Fair)
    prob_fair = 1.0
    for char in seq:
        if char == 'H':
            prob_fair *= p_f
        else:
            prob_fair *= (1 - p_f)
            
    if prob_biased > prob_fair:
        print("Q1: Biased")
    else:
        print("Q1: Fair")

# Q3 Consider an HMM with three states that emits a string of length four from an alphabet of three symbols.
# Compute the number of edges in the Viterbi graph corresponding to this HMM (don't forget to include the source and sink nodes).
def q3_solver():
    # HMM with k=3 states, string length n=4
    # Edges in Viterbi graph?
    # Standard Viterbi graph structure:
    # Source -> Col 1 (k edges)
    # Col 1 -> Col 2 (k*k edges)
    # ...
    # Col n-1 -> Col n (k*k edges)
    # Col n -> Sink (k edges)
    
    states = 3
    length = 4
    
    # Edges from Source to first column
    edges_source = states
    
    # Edges between columns (n-1 transitions)
    edges_internal = (length - 1) * (states * states)
    
    # Edges from last column to Sink
    edges_sink = states
    
    total_edges = edges_source + edges_internal + edges_sink
    print(f"Q3: {total_edges}")

# Q4 Say that we are in a crooked casino where the probability of heads for the biased coin is 3/4.
# Given the sequence of coins π = BFBBF and the sequence of coin flips x = HTHHH, compute the probability Pr(x|π) that this sequence was generated by the given states.
# Round your answer to three decimal places.
def q4_solver():
    # P(H|B)=0.75, P(T|B)=0.25
    # P(H|F)=0.5, P(T|F)=0.5
    path = "BFBBF"
    seq = "HTHHH"
    
    prob = 1.0
    for i in range(len(seq)):
        state = path[i]
        symbol = seq[i]
        
        if state == 'B':
            p_emit = 0.75 if symbol == 'H' else 0.25
        else: # Fair
            p_emit = 0.5 if symbol == 'H' else 0.5
            
        prob *= p_emit
        
    print(f"Q4: {prob:.3f}")

# Q5 Say that we are given the following amino acid alignment (columns with insertion rate >θ already removed).
# What is the probability that serine (S) will be emitted from M(8), the 8th match state of the profile HMM corresponding to this alignment? (Write your answer as a decimal.)
def q5_solver():
    alignment_str = """
M--QKCASHLE-AR
MSNL-C-APD-LER
MSAPNCARKYDI-R
MS-SSCADED-IIR
M--TKC-SKLEIDR
""".strip().split('\n')
    
    # M(8) corresponds to the 8th column (0-indexed 7)
    # Assuming the provided alignment contains only match columns and deletions (gaps).
    col_idx = 7
    symbol_to_count = 'S'
    
    col_chars = []
    for row in alignment_str:
        if len(row) > col_idx:
            char = row[col_idx]
            # In Profile HMM emission calculations, we only consider symbols emitted by Match state.
            # Gaps in a match column correspond to Delete state, so they don't contribute to emission counts.
            if char != '-':
                col_chars.append(char)
                
    count = col_chars.count(symbol_to_count)
    total = len(col_chars)
    
    if total > 0:
        prob = count / total
        print(f"Q5: {prob}")
    else:
        print("Q5: No emissions in column 8")

print("--- Quiz Solutions ---")
q1_solver()
q3_solver()
q4_solver()
q5_solver()

--- Quiz Solutions ---
Q1: Fair
Q3: 33
Q4: 0.105
Q5: 0.4
